# Adversial Loop

In [126]:
from peft import PeftModel
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
import unicodedata
import re

In [127]:
!rm -rf bert-lora/
!rm -rf gpt-lora/

In [128]:
base_attack = GPT2LMHeadModel.from_pretrained("gpt2-medium")
tok_att = GPT2Tokenizer.from_pretrained("gpt2-medium")
tok_att.pad_token = tok_att.eos_token

!git clone https://huggingface.co/kunjcr2/gpt-lora
attacker = PeftModel.from_pretrained(base_attack, "./gpt-lora")
attacker = attacker.merge_and_unload()

!git clone https://huggingface.co/kunjcr2/bert-lora
base_defend = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tok_def = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
defender = PeftModel.from_pretrained(base_defend, "./bert-lora")
defender = defender.merge_and_unload()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cloning into 'gpt-lora'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 8), reused 0 (delta 0), pack-reused 3 (from 1)
Receiving objects: 100% (34/34), 11.46 KiB | 11.46 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Cloning into 'bert-lora'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 75 (delta 20), reused 0 (delta 0), pack-reused 14 (from 1)
Receiving objects: 100% (75/75), 227.60 KiB | 628.00 KiB/s, done.
Resolving deltas: 100% (22/22), done.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [130]:
def get_bypasses(texts, defn, tok_defn, device="cuda", batch_size=64):
    """Returns texts where defn predicts benign (label=0) — i.e. bypasses."""
    defn.eval().to(device)
    results = []
    n = len(text)
    print(f"Total: {n}, Batches: {n//batch_size + 1}")

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok_defn(batch, truncation=True, padding=True,
                      max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = defn(**enc).logits
        preds = logits.argmax(-1).cpu().tolist()
        results.extend([(t, p) for t, p in zip(batch, preds)])

        print(f"Batch {i//batch_size} Done")

    return [t for t, p in results if p == 0]  # fooled the defn

In [131]:
def generate_evasions(n_total, att, tok_att, batch_size=64, device="cuda"):
    att.eval().to(device)
    evasions = []
    prompt = "Injection Prompt in English: "
    print(f"Total: {n_total//batch_size+1} batches")

    for i in range(0, n_total, batch_size):
        prompts = [prompt] * min(batch_size, n_total - i)
        enc = tok_att(prompts, return_tensors="pt", padding=True,
                      truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = att.generate(
                **enc,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.6,
                top_p=0.95,
                pad_token_id=tok_att.eos_token_id,
            )
        for seq in out:
            full_text = tok_att.decode(seq, skip_special_tokens=True)
            clean = full_text.split("Injection Prompt in English: ")[-1].strip()
            if clean:
                evasions.append(clean)

        print(f"Batch: {i//batch_size} up")

    return evasions

In [132]:
with open("/content/up_mal.json") as f:
  data = json.loads(f.read())

text = list(map(lambda x: x["user"], data))

In [144]:
evasions = generate_evasions(1000, attacker, tok_att) # getting the evasions (new mal prompts)

Total: 16 batches
Batch: 0 up
Batch: 1 up
Batch: 2 up
Batch: 3 up
Batch: 4 up
Batch: 5 up
Batch: 6 up
Batch: 7 up
Batch: 8 up
Batch: 9 up
Batch: 10 up
Batch: 11 up
Batch: 12 up
Batch: 13 up
Batch: 14 up
Batch: 15 up


In [145]:
def is_clean(text, min_words=5, max_non_ascii_ratio=0.3):
    if len(text.split()) < min_words:
        return False
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / len(text) > max_non_ascii_ratio:
        return False
    return True

def clean_evasion(text):
    # Strip leading punctuation/symbol artifacts
    text = re.sub(r'^[\W_]+', '', text).strip()
    # Normalize excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    return text

evasions = [clean_evasion(e) for e in evasions if is_clean(e)]

In [146]:
len(evasions)

589

In [147]:
bypasses = get_bypasses(evasions, defender, tok_def) # seeing if those new mal prompts can bypass

Total: 7110, Batches: 112
Batch 0 Done
Batch 1 Done
Batch 2 Done
Batch 3 Done
Batch 4 Done
Batch 5 Done
Batch 6 Done
Batch 7 Done
Batch 8 Done
Batch 9 Done


In [148]:
bypass_rate_r1 = len(bypasses) / len(evasions)
print(f"Bypass rate: {bypass_rate_r1:.2%}")

Bypass rate: 2.38%


You want bypass rate **low**:

- Base model: 2% bypass → not because it's good, but because it's predicting everything as benign, so "bypassing" is meaningless
- Your v1 defender: 23% bypass → it's actually catching injections, but 23% still slip through
- Your v2 defender: 3.72% bypass → genuinely better, it learned the evasion patterns

The base model has no concept of what an injection is. It's not catching anything, so nothing can bypass it.

# Retrain BERT

In [94]:
import pandas as pd, json, random, torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

# ── Load original benign data ─────────────────────────────────────
with open("/content/benign.json") as f:
    benign = json.loads(f.read())

# ── Build fine-tune dataset ───────────────────────────────────────
bypass_examples = [{"user": text, "label": 1} for text in bypasses]
benign_sample   = random.sample(benign, min(len(bypasses) * 2, len(benign)))
combined        = bypass_examples + benign_sample

df = pd.DataFrame(combined)
X_train, X_test, y_train, y_test = train_test_split(
    df["user"], df["label"],
    test_size=0.1, random_state=42, stratify=df["label"]
)

train_ds = Dataset.from_pandas(pd.concat([X_train, y_train], axis=1))
test_ds  = Dataset.from_pandas(pd.concat([X_test,  y_test],  axis=1))

# ── Load existing trained defender ────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
base      = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased")
model_v2  = PeftModel.from_pretrained(base, "kunjcr2/bert-lora")  # start from v1 weights
for name, param in model_v2.named_parameters():
    if "lora" in name:
        param.requires_grad = True

def tokenize(ex):
    return tokenizer(ex["user"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True).rename_column("label", "labels")
test_ds  = test_ds.map(tokenize,  batched=True).rename_column("label", "labels")

# ── Training args ─────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./bert-lora",
    per_device_train_batch_size=32,
    num_train_epochs=3,
    learning_rate=1e-4,          # lower LR — fine-tuning on top of v1
    warmup_steps=10,
    optim="adamw_torch_fused",
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.95,
    gradient_accumulation_steps=2,
    fp16=True,
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    report_to=["wandb"],
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id="kunjcr2/bert-lora",
    hub_strategy="every_save",
)

trainer = Trainer(
    model=model_v2,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)
trainer.train()

# ── Push with v2 commit message ───────────────────────────────────
model_v2.push_to_hub(
    "kunjcr2/bert-lora",
    commit_message="v2: adversarial fine-tune on bypass examples"
)
tokenizer.push_to_hub(
    "kunjcr2/bert-lora",
    commit_message="v2: adversarial fine-tune on bypass examples"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/391 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


README.md:   0%|          | 0.00/726 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  73%|#######3  | 3.91MB / 5.33MB            

CommitInfo(commit_url='https://huggingface.co/kunjcr2/bert-lora/commit/79710f80e27fbab95c69e6b5d2b2734f245858f7', commit_message='v2: adversarial fine-tune on bypass examples', commit_description='', oid='79710f80e27fbab95c69e6b5d2b2734f245858f7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kunjcr2/bert-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='kunjcr2/bert-lora'), pr_revision=None, pr_num=None)